# Ames Housing: Data Exploration
**Team:** Standard Deviants

This notebook loads the Ames Housing dataset, cleans it, and saves a cleaned copy for the descriptive statistics, visualizations, and regression notebooks.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("../data/AmesHousing.csv")

In [4]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Rows: 2930, Columns: 82


In [5]:
pd.set_option("display.max_rows", 100)
df.dtypes

Order                int64
PID                  int64
MS SubClass          int64
MS Zoning           object
Lot Frontage       float64
Lot Area             int64
Street              object
Alley               object
Lot Shape           object
Land Contour        object
Utilities           object
Lot Config          object
Land Slope          object
Neighborhood        object
Condition 1         object
Condition 2         object
Bldg Type           object
House Style         object
Overall Qual         int64
Overall Cond         int64
Year Built           int64
Year Remod/Add       int64
Roof Style          object
Roof Matl           object
Exterior 1st        object
Exterior 2nd        object
Mas Vnr Type        object
Mas Vnr Area       float64
Exter Qual          object
Exter Cond          object
Foundation          object
Bsmt Qual           object
Bsmt Cond           object
Bsmt Exposure       object
BsmtFin Type 1      object
BsmtFin SF 1       float64
BsmtFin Type 2      object
B

In [6]:
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


## Data Cleaning (Issue #5)
Check for missing values, duplicates, and data-type issues. The original `AmesHousing.csv` is never modified; the cleaned data is saved as a new file.

In [8]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate PIDs:", df["PID"].duplicated().sum())

Duplicate rows: 0
Duplicate PIDs: 0


In [9]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

Pool QC           2917
Misc Feature      2824
Alley             2732
Fence             2358
Mas Vnr Type      1775
Fireplace Qu      1422
Lot Frontage       490
Garage Cond        159
Garage Qual        159
Garage Finish      159
Garage Yr Blt      159
Garage Type        157
Bsmt Exposure       83
BsmtFin Type 2      81
Bsmt Cond           80
Bsmt Qual           80
BsmtFin Type 1      80
Mas Vnr Area        23
Bsmt Half Bath       2
Bsmt Full Bath       2
BsmtFin SF 1         1
Garage Cars          1
Garage Area          1
Total Bsmt SF        1
Bsmt Unf SF          1
BsmtFin SF 2         1
Electrical           1
dtype: int64

**Cleaning decisions:**
- Most missing values mean the house doesn't have that feature (no pool, no garage, no basement, no alley, etc.), from the dataset documentation. Categorical ones are filled with `"None"` and numeric ones with `0`.
- `Lot Frontage` is missing, so it is filled with the median.
- One row is missing `Electrical`, so it is dropped.
- `Garage Yr Blt` is left blank for houses with no garage, because there is no year to fill in.
- The dataset author recommends removnig the 5 houses over 4,000 sq ft (`Gr Liv Area`), they are outliers.
- `MS SubClass` is a category code, so it is converted to text.
- Spaces and `/` are removed from column names (like Gr Liv Area → GrLivArea) to make them easier to use in our regression formulas.

In [11]:
df_clean = df.copy()
none_cols = ["Pool QC", "Misc Feature", "Alley", "Fence", "Mas Vnr Type", "Fireplace Qu",
             "Garage Type", "Garage Finish", "Garage Qual", "Garage Cond",
             "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]
df_clean[none_cols] = df_clean[none_cols].fillna("None")
zero_cols = ["Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF",
             "Bsmt Full Bath", "Bsmt Half Bath", "Garage Cars", "Garage Area"]
df_clean[zero_cols] = df_clean[zero_cols].fillna(0)
df_clean["Lot Frontage"] = df_clean["Lot Frontage"].fillna(df_clean["Lot Frontage"].median())
df_clean = df_clean.dropna(subset=["Electrical"])
df_clean = df_clean[df_clean["Gr Liv Area"] <= 4000]
df_clean["MS SubClass"] = df_clean["MS SubClass"].astype(str)
df_clean.columns = df_clean.columns.str.replace(r"[ /]", "", regex=True)

**Selected predictors:** `OverallQual`, `GrLivArea`, `TotalBsmtSF`, and `YearBuilt` are our independent variables for predicting `SalePrice`.

In [13]:
predictors = ["OverallQual", "GrLivArea", "TotalBsmtSF", "YearBuilt"]
df_clean[predictors + ["SalePrice"]].corr()["SalePrice"].round(2)

OverallQual    0.81
GrLivArea      0.72
TotalBsmtSF    0.66
YearBuilt      0.57
SalePrice      1.00
Name: SalePrice, dtype: float64

In [14]:
df_clean.to_csv("../data/AmesHousing_clean.csv", index=False)